# dARK Admin API - Test Notebook

This notebook tests the `dark-core-admin-api` endpoints using FastAPI's TestClient.

It covers:
1. **Setup**: Initializing the Orchestrator and TestClient.
2. **Health Check**: Verifying API availability and admin status.
3. **Authority Management**: Registering, querying, and managing authorities.
4. **Wallet Operations**: Funding wallets and checking balances.

**Note**: This API is for administrative operations only. It should run on a dedicated Admin Node.

## 1. Setup Environment

In [ ]:
import os
import sys
import logging
from dotenv import load_dotenv

# Add app to path
sys.path.append(os.path.abspath('..'))

# Logging
logging.basicConfig(level=logging.INFO)

# Load unified configuration
if os.path.exists('../../.env'):
    load_dotenv('../../.env')
    print("Loaded ../../.env")
elif os.path.exists('../.env'):
    load_dotenv('../.env')
    print("Loaded ../.env")
else:
    print('⚠️  Warning: No .env found')

## 2. Initialize App & TestClient

In [ ]:
from fastapi.testclient import TestClient
from app.main import app
from app.dependencies import get_orchestrator, init_orchestrator

# Initialize the orchestrator manually before creating TestClient
try:
    orchestrator = get_orchestrator()
    print("✅ Orchestrator already initialized")
except RuntimeError:
    print("Initializing orchestrator manually...")
    init_orchestrator()
    print("✅ Orchestrator initialized")

# Create TestClient
client = TestClient(app)

print("✅ TestClient initialized")

## 3. Health Check

In [ ]:
response = client.get("/health")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")
assert response.status_code == 200

## 4. Get Admin Status

Check the admin account status and blockchain connection.

In [ ]:
response = client.get("/api/v1/admin/status")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    data = response.json()
    print(f"\n📊 Admin Status:")
    print(f"   Admin Address: {data['admin_address']}")
    print(f"   Admin Balance: {data['admin_balance_eth']} ETH")
    print(f"   Connected: {data['blockchain_connected']}")
    print(f"   Block: {data['current_block']}")
    print(f"   Chain ID: {data['chain_id']}")

## 5. Register a New Authority

Create a new authority with wallet, register on blockchain, and authorize NAANs.

In [ ]:
import time

# Generate unique UUID for testing
TEST_UUID = f"admin-test-{int(time.time())}"
TEST_NAANS = ["12345", "67890"]

payload = {
    "uuid": TEST_UUID,
    "naans": TEST_NAANS,
    "fund_amount_eth": 0.01
}

print(f"Registering authority: {TEST_UUID}")
print(f"NAANs: {TEST_NAANS}")

response = client.post("/api/v1/admin/authority", json=payload)
print(f"\nStatus Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    data = response.json()
    print(f"\n✅ Authority registered successfully!")
    print(f"   UUID: {data['uuid']}")
    print(f"   Wallet: {data['wallet_address']}")
    print(f"   NAANs: {data['naans']}")
    print(f"   Balance: {data['balance_eth']} ETH")

## 6. Get Authority Info

Retrieve information about the authority we just created.

In [ ]:
response = client.get(f"/api/v1/admin/authority/{TEST_UUID}")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    data = response.json()
    assert data['uuid'] == TEST_UUID
    assert set(TEST_NAANS).issubset(set(data['naans']))
    print("\n✅ Authority info retrieved successfully!")

## 7. Authorize Additional NAAN

Add a new NAAN to the authority.

In [ ]:
NEW_NAAN = "99999"

payload = {"naan": NEW_NAAN}

print(f"Authorizing NAAN {NEW_NAAN} for {TEST_UUID}...")

response = client.post(f"/api/v1/admin/authority/{TEST_UUID}/authorize-naan", json=payload)
print(f"\nStatus Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    data = response.json()
    print(f"\n✅ {data['message']}")

## 8. Check Wallet Balance

In [ ]:
response = client.get(f"/api/v1/admin/authority/{TEST_UUID}/balance")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    data = response.json()
    print(f"\n💰 Wallet Balance:")
    print(f"   UUID: {data['uuid']}")
    print(f"   Wallet: {data['wallet_address']}")
    print(f"   Balance: {data['balance_eth']} ETH")

## 9. Fund Authority Wallet

Send additional ETH to the authority's wallet.

In [ ]:
payload = {"amount_eth": 0.005}

print(f"Funding {TEST_UUID} with 0.005 ETH...")

response = client.post(f"/api/v1/admin/authority/{TEST_UUID}/fund", json=payload)
print(f"\nStatus Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    data = response.json()
    print(f"\n✅ {data['message']}")
    if data.get('transaction_hash'):
        print(f"   TX Hash: {data['transaction_hash']}")

## 10. Verify Updated Balance

In [ ]:
response = client.get(f"/api/v1/admin/authority/{TEST_UUID}/balance")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

if response.status_code == 200:
    data = response.json()
    print(f"\n💰 Updated Balance: {data['balance_eth']} ETH")

## 11. Test Error Handling - Authority Not Found

In [ ]:
response = client.get("/api/v1/admin/authority/non-existent-uuid")
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

assert response.status_code == 404
print("\n✅ 404 returned correctly for non-existent authority")

## 12. Test Error Handling - Duplicate Authority

In [ ]:
# Try to register the same authority again
payload = {
    "uuid": TEST_UUID,
    "naans": ["12345"]
}

response = client.post("/api/v1/admin/authority", json=payload)
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

assert response.status_code == 409
print("\n✅ 409 returned correctly for duplicate authority")

## Summary

All Admin API endpoints tested successfully:

| Endpoint | Method | Status |
|----------|--------|--------|
| `/health` | GET | ✅ |
| `/api/v1/admin/status` | GET | ✅ |
| `/api/v1/admin/authority` | POST | ✅ |
| `/api/v1/admin/authority/{uuid}` | GET | ✅ |
| `/api/v1/admin/authority/{uuid}/authorize-naan` | POST | ✅ |
| `/api/v1/admin/authority/{uuid}/balance` | GET | ✅ |
| `/api/v1/admin/authority/{uuid}/fund` | POST | ✅ |